<a href="https://colab.research.google.com/github/LineytsevE/LeraAssistant/blob/main/piper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get update
!apt-get install -y build-essential cmake ninja-build

In [ ]:
!git clone https://github.com/OHF-voice/piper1-gpl.git
%cd piper1-gpl

In [ ]:
!python3 -m venv .venv --without-pip
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!.venv/bin/python3 get-pip.py

In [ ]:
!.venv/bin/pip install --upgrade pip
!.venv/bin/pip install -e .[train]

In [ ]:
!bash build_monotonic_align.sh

In [ ]:
!.venv/bin/pip install scikit-build

In [ ]:
!.venv/bin/python setup.py build_ext --inplace

In [ ]:
!cp -r /content/drive/MyDrive/espeak-ng-data/* /content/piper1-gpl/src/piper/espeak-ng-data/

In [ ]:
!.venv/bin/python -m piper.train fit \
  --data.voice_name "ru_RU-kuznets-medium" \
  --data.csv_path "/content/drive/MyDrive/glados20/metadata.csv" \
  --data.audio_dir "/content/drive/MyDrive/glados20/" \
  --model.sample_rate 22050 \
  --data.espeak_voice "ru" \
  --data.cache_dir "/content/cache/" \
  --data.config_path "/content/drive/MyDrive/ru_RU-glados-medium.json" \
  --data.batch_size 8 \
  --ckpt_path "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU/irina/medium/epoch%3D4139-step%3D929464.ckpt" \
  --trainer.check_val_every_n_epoch 50 \
  --trainer.log_every_n_steps 1 \
  --data.num_workers 2

In [ ]:
!.venv/bin/python -m piper.train.export_onnx \
  --checkpoint /content/piper1-gpl/lightning_logs/version_0/checkpoints/epoch=*-step=*.ckpt \
  --output-file /content/drive/MyDrive/ru_RU-sushko-medium.onnx